# NB1 - The Evaluation Interface (V) and Measuring Your Agent

**Workshop: Self-Evolving Agents by Optimizing the Harness (no GPU)**

In **NB0** you built an agent: brain + context (**C**) + tool (**T**) + execution
loop (**E**). The one thing it's missing is a way to know **how good it is**.
That's component **V** - the evaluation interface - and it's the job of this
notebook.

> **Thesis of the day:** *Reflection is the gradient, the skill document is the
> parameter vector, and your eval set is the loss.* No loss -> no learning. So we
> start with the loss.

In this notebook we:
1. Meet the **environment** (the text-to-SQL task) and the eval set.
2. Build the **reward signal** `score_sql` (execution match - objective, automatic).
3. **Measure the agent from NB0** to get a baseline number.
4. Do error analysis - the failures are the raw material every later notebook
   learns from.

In [ ]:
# Setup. We run from the notebooks/ folder, so add the repo root to the path.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from workshop_utils import (
    build_db, load_tasks, run_sql, score_sql, evaluate,
    llm, METER, SCHEMA_TEXT, extract_sql, baseline_prompt, make_agent,
)

DB = build_db()          # deterministic rebuild; same data on every machine
print("Database ready at:", DB)

## 1. The environment: a toy "shop" database

The agent's job: translate a natural-language question into a **SQLite SELECT**
that returns the right rows. The schema is small enough to fit in a prompt and
rich enough to need joins, group-bys, subqueries, and date handling.

In [ ]:
print(SCHEMA_TEXT)
print("Sample customers:", run_sql("SELECT * FROM customers LIMIT 3")[0])
print("Sample orders:   ", run_sql("SELECT * FROM orders LIMIT 3")[0])
print("Sample products: ", run_sql("SELECT * FROM products LIMIT 3")[0])

## 2. The eval set (NL -> gold SQL)

40 tasks, labelled `easy | medium | hard`, split into **train** (we may optimize
on these) and **test** (held out - the number we actually trust).

The train/test split is the single most important discipline in the workshop:
**a self-evolving agent must improve on train without ever peeking at test.**

In [ ]:
from collections import Counter
tasks = load_tasks()
print("total tasks:", len(tasks))
print("split:", dict(Counter(t["split"] for t in tasks)))
print("level:", dict(Counter(t["level"] for t in tasks)))
print()
for t in tasks[:3] + tasks[20:22]:
    print(f"#{t['id']:>2} [{t['split']}/{t['level']}] {t['question']}")
    print(f"     gold: {t['gold']}")

## 3. The reward signal V: `score_sql` (execution match)

We do **not** compare SQL strings - there are many correct ways to write the same
query. Instead we *execute* both the predicted and the gold query and compare the
**result sets**. Order matters only when the gold query uses `ORDER BY`.

This is exactly the "decomposed verifiable reward" idea from the ASG-SI paper:
an objective, replayable check, not an LLM's opinion.

In [ ]:
t = next(x for x in tasks if x["id"] == 1)
print("Q:", t["question"])
print("gold vs gold  ->", score_sql(t["gold"], t["gold"]))                    # True
print("wrong query   ->", score_sql("SELECT city FROM customers", t["gold"]))  # False
print("syntax error  ->", score_sql("SELECT nope FROM nope", t["gold"]))       # False

## 4. Measure the agent from NB0

`make_agent()` is the exact harness you built in NB0 (context **C** + tool **T** +
execution loop **E**). No memory, no examples, no skills - the bare agent. This
is the **baseline**: the number every later notebook must beat *without touching
the weights*, only by evolving the harness.

In [ ]:
agent = make_agent()      # the NB0 agent: generate -> run -> repair-on-error

# The entire "harness" the brain sees right now is just this prompt:
print(baseline_prompt("How many customers are there in total?")[1]["content"])

### Run the agent on the held-out test split

This makes real API calls with **your** key. 16 test tasks; a few cents on
`gpt-4o-mini`. The cost meter prints the spend. (The repair loop may add a call
or two when a query first fails to execute.)

In [ ]:
METER.reset()
baseline = evaluate(agent, split="test", verbose=True)
print()
print("TEST accuracy:", round(baseline["accuracy"], 3))
print("by level:    ", {k: round(v["acc"], 2) for k, v in baseline["by_level"].items()})
print(METER)

## 5. Error analysis - the fuel for self-evolution

Every failure below is a learning signal. Notice the *kind* of failure: the
execution loop already eliminated the crashes, so what remains are queries that
**run fine but return the wrong rows** - subtly wrong joins, a missing
`status='completed'` filter, the wrong `GROUP BY`. Robustness can't catch those;
only *learning* can. That gap is exactly what NB2 onward fills.

In [ ]:
fails = [r for r in baseline["records"] if not r["correct"]]
print(f"{len(fails)} failures on test\n")
for r in fails:
    print(f"# {r['id']} [{r['level']}] {r['question']}")
    print("  pred:", r["pred"])
    print("  gold:", r["gold"])
    print()

In [ ]:
# Save the baseline number so later notebooks can show the lift over it.
import json
os.makedirs("../data", exist_ok=True)
with open("../data/baseline_test.json", "w", encoding="utf-8") as f:
    json.dump(
        {"accuracy": baseline["accuracy"],
         "by_level": {k: v["acc"] for k, v in baseline["by_level"].items()}},
        f, indent=2,
    )
print("saved baseline_test.json")

## Takeaways

- The **eval interface (V)** is the foundation of self-evolution. Without an
  objective reward, "self-improvement" is just vibes.
- Execution-match is a clean, replayable reward - no GPU, no LLM judge.
- Your **NB0 agent** is now measured. Everything from here on raises that test
  number by changing the *harness*, never the weights.
- The remaining failures aren't crashes - they're *subtly wrong* answers. You
  can't fix those with a retry; the agent has to **learn**.

### Exercise
1. Re-run the baseline with `temperature=0.7` (edit `llm(...)`). Does accuracy
   change? What does that tell you about prompt vs. sampling?
2. Add one new hard question + gold SQL to `workshop_utils/tasks.py` and re-run.

**Next - NB2:** give the agent a memory (**S**) and let it learn from these
failures. *Reflection is the gradient.*